# SHAP Visualizations for Latest Per-Market Run

This notebook reads SHAP outputs generated from saved models and renders quick visual diagnostics.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_colwidth', 120)

In [ ]:
RUN_DIR = Path('/Users/simonbisgaard/Desktop/nitor/runs/20260220-153811_per_market_interactions')

global_imp = pd.read_csv(RUN_DIR / 'global_feature_importance_shap.csv')
local_imp = pd.read_csv(RUN_DIR / 'local_feature_importance_shap.csv')
global_rows = pd.read_csv(RUN_DIR / 'global_shap_sample_rows.csv')

print('Run:', RUN_DIR)
print('global_imp:', global_imp.shape)
print('local_imp:', local_imp.shape)
print('global_rows:', global_rows.shape)

In [ ]:
top_n = 20
top = global_imp.head(top_n).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top['feature'], top['mean_abs_shap'], color='#2a9d8f')
ax.set_title(f'Global SHAP Feature Importance (Top {top_n})')
ax.set_xlabel('mean(|SHAP value|)')
ax.set_ylabel('feature')
plt.tight_layout()
plt.show()

global_imp.head(20)

In [ ]:
# Beeswarm-like SHAP distribution (x=SHAP, y=feature rank)
top_n = 12
top_features = global_imp.head(top_n)['feature'].tolist()
plot_df = []
rng = np.random.default_rng(42)

for i, feat in enumerate(top_features):
    col = f'shap__{feat}'
    vals = global_rows[col].dropna().to_numpy()
    if len(vals) > 1500:
        vals = rng.choice(vals, size=1500, replace=False)
    y = np.full_like(vals, fill_value=top_n - i, dtype=float) + rng.normal(0, 0.08, size=len(vals))
    plot_df.append(pd.DataFrame({'feature': feat, 'rank': top_n - i, 'shap': vals, 'y': y}))

plot_df = pd.concat(plot_df, ignore_index=True)

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(plot_df['shap'], plot_df['y'], s=8, alpha=0.35, color='#264653', edgecolors='none')
ax.axvline(0.0, color='black', linewidth=1)
ax.set_yticks(range(1, top_n + 1))
ax.set_yticklabels(top_features[::-1])
ax.set_xlabel('SHAP value')
ax.set_ylabel('feature')
ax.set_title('SHAP Distribution (Top Features)')
plt.tight_layout()
plt.show()

In [ ]:
# Top features per market
markets = sorted(local_imp['market'].unique())
top_k = 10
n = len(markets)
fig, axes = plt.subplots(nrows=n, ncols=1, figsize=(10, 3.2 * n), sharex=False)
if n == 1:
    axes = [axes]

for ax, market in zip(axes, markets):
    m = local_imp[local_imp['market'] == market].nlargest(top_k, 'mean_abs_shap').iloc[::-1]
    ax.barh(m['feature'], m['mean_abs_shap'], color='#e76f51')
    ax.set_title(f'{market} - Top {top_k} Local SHAP Features')
    ax.set_xlabel('mean(|SHAP value|)')

plt.tight_layout()
plt.show()

In [ ]:
# Single-row explanation helper
def explain_row(row_idx: int = 0, top_k: int = 12):
    r = global_rows.iloc[row_idx]
    shap_cols = [c for c in global_rows.columns if c.startswith('shap__')]
    vals = pd.Series({c.replace('shap__', ''): r[c] for c in shap_cols})
    top = vals.abs().nlargest(top_k).index
    d = vals.loc[top].sort_values()

    fig, ax = plt.subplots(figsize=(10, 5.5))
    colors = ['#d62828' if v < 0 else '#2a9d8f' for v in d.values]
    ax.barh(d.index, d.values, color=colors)
    ax.axvline(0.0, color='black', linewidth=1)
    ax.set_title(f'Row {row_idx} SHAP Contributions | prediction={r["prediction"]:.3f} | base={r["base_value"]:.3f}')
    ax.set_xlabel('SHAP value contribution')
    plt.tight_layout()
    plt.show()

    display(pd.DataFrame({
        'meta': ['id', 'market', 'delivery_start', 'target', 'base_value', 'prediction'],
        'value': [r.get('id'), r.get('market'), r.get('delivery_start'), r.get('target'), r.get('base_value'), r.get('prediction')]
    }))

explain_row(0, 12)